# Tiền xử lý dữ liệu TMDB Movies

## 1. Mục tiêu tiền xử lý

Dataset gốc là dữ liệu phim từ TMDB (The Movie Database). Nhóm tập trung vào phim có quốc gia sản xuất thuộc **Việt Nam**, **Hàn Quốc** và **Trung Quốc** để so sánh đặc điểm, mức độ phổ biến và đánh giá.

**Phạm vi bước này:**
- Lọc dữ liệu theo quốc gia sản xuất (`production_countries`).
- Lọc theo năm phát hành 2000–2025 (mở rộng từ 2000 để tăng số lượng phim Việt Nam).
- Chỉ giữ phim có dữ liệu chấm điểm hợp lệ và trạng thái đã phát hành.
- Đổi tên cột sang tiếng Việt không dấu phục vụ Power BI.
- Bổ sung các cờ kiểm tra chất lượng dữ liệu.

**Lưu ý quan trọng:**
- Đây là bước **clean/filter tổng ban đầu**, chưa phải bước tách bảng quan hệ cuối cùng cho Power BI.
- Các bước chuẩn hóa sâu và tách bảng Dim/Fact/Bridge sẽ thực hiện ở bước riêng sau.
- Với Trung Quốc, tạm thời chỉ lấy `China`, không lấy `Hong Kong`, `Taiwan`, `Macau`.
- Một phim có thể có nhiều quốc gia sản xuất (phim đồng sản xuất). Nên diễn đạt là "phim có quốc gia sản xuất thuộc Việt Nam/Hàn Quốc/Trung Quốc" thay vì khẳng định tuyệt đối là "phim nội địa".
- Khi so sánh giữa các quốc gia, không chỉ dùng số lượng tuyệt đối mà nên dùng thêm tỷ lệ, trung bình, top phim hoặc phân tích theo từng nhóm.
- Dữ liệu năm 2025 có thể chưa đầy đủ vì chưa phản ánh trọn năm, nên khi nhận xét xu hướng theo thời gian cần cẩn thận.
- Doanh thu và ngân sách thường thiếu nhiều trong TMDB, nên chỉ dùng cho phân tích phụ trên nhóm phim có dữ liệu hợp lệ.
- Không xóa dòng chỉ vì thiếu doanh thu, ngân sách, thời lượng, tóm tắt nội dung hoặc câu giới thiệu.

## 2. Import thư viện và cấu hình đường dẫn

Nạp các thư viện cần dùng, khai báo đường dẫn dữ liệu gốc và đường dẫn file kết quả. Thư mục `data/processed` được tạo tự động nếu chưa tồn tại.

In [ ]:
import ast
import json
import os
import re

import pandas as pd

# Khai báo đường dẫn dữ liệu đầu vào và đầu ra.
RAW_DATA_PATH = "data/raw/TMDB_movie_dataset.csv"
OUTPUT_PATH = "data/processed/tmdb_vn_kr_cn_2000_2025_scored.csv"

# Tạo thư mục lưu dữ liệu đã xử lý nếu chưa có.
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

## 3. Đọc dữ liệu thô và kiểm tra tổng quan

Đọc file CSV gốc, kiểm tra kích thước dữ liệu, danh sách cột và một vài dòng đầu. Notebook cũng kiểm tra các cột bắt buộc trước khi lọc.

In [ ]:
# Đọc dữ liệu gốc từ file CSV.
df = pd.read_csv(RAW_DATA_PATH)

print(f"Shape ban đầu: {df.shape}")
print("\nDanh sách cột:")
print(df.columns.tolist())

display(df.head())

# Kiểm tra các cột bắt buộc cho phạm vi lọc ban đầu.
required_columns = [
    "production_countries",
    "release_date",
    "vote_average",
    "vote_count",
]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Thiếu các cột bắt buộc: {missing_columns}")
else:
    print("\n✅ Tất cả các cột bắt buộc đều tồn tại.")

## 4. Lọc theo quốc gia sản xuất

Lọc theo cột `production_countries`, không lọc theo `original_language`. Với Trung Quốc, chỉ lấy `China` và tạm thời không lấy `Hong Kong`, `Taiwan`, `Macau`.

Dữ liệu `production_countries` có thể ở dạng text thường, list, chuỗi JSON-like hoặc null nên cần hàm xử lý chắc chắn.

In [ ]:
TARGET_COUNTRIES = ["Vietnam", "South Korea", "China"]
TARGET_COUNTRY_SET = set(TARGET_COUNTRIES)


def _is_missing(value):
    """Kiểm tra giá trị thiếu mà không làm lỗi với list/dict."""
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    return False


def _parse_json_like_text(value):
    """Thử chuyển chuỗi JSON/list/dict thành object Python."""
    if not isinstance(value, str):
        return value

    text = value.strip()
    if not text:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
            continue

    return text


def extract_country_names(value):
    """Trích xuất tên quốc gia từ null, text, list hoặc chuỗi JSON-like."""
    if _is_missing(value):
        return []

    parsed_value = _parse_json_like_text(value)
    country_names = []

    def collect_country_name(item):
        if _is_missing(item):
            return

        if isinstance(item, dict):
            name = item.get("name") or item.get("country") or item.get("english_name")
            if name:
                country_names.append(str(name).strip())
            return

        if isinstance(item, list):
            for child_item in item:
                collect_country_name(child_item)
            return

        if isinstance(item, str):
            text = item.strip()
            if not text:
                return

            parsed_text = _parse_json_like_text(text)
            if parsed_text is not text:
                collect_country_name(parsed_text)
                return

            # Tách chuỗi văn bản thường theo các dấu phân cách phổ biến.
            for name in re.split(r"[,;/|]", text):
                clean_name = name.strip()
                if clean_name:
                    country_names.append(clean_name)

    collect_country_name(parsed_value)

    # Loại trùng nhưng giữ nguyên thứ tự xuất hiện.
    unique_country_names = []
    seen = set()
    for name in country_names:
        normalized_name = name.casefold()
        if normalized_name not in seen:
            unique_country_names.append(name)
            seen.add(normalized_name)

    return unique_country_names


def get_selected_countries(value):
    """Chỉ giữ các quốc gia thuộc phạm vi nghiên cứu."""
    country_names = extract_country_names(value)
    selected = [c for c in country_names if c in TARGET_COUNTRY_SET]
    return selected


# Tạo các cột quốc gia được chọn để phục vụ lọc và phân tích sau này.
country_df = df.copy()
country_df["selected_countries"] = country_df["production_countries"].apply(get_selected_countries)
country_df["selected_countries_text"] = country_df["selected_countries"].apply(
    lambda countries: ", ".join(countries)
)
country_df["primary_selected_country"] = country_df["selected_countries"].apply(
    lambda countries: countries[0] if countries else pd.NA
)

# Chỉ giữ phim có ít nhất một quốc gia thuộc Vietnam, South Korea hoặc China.
filtered_country_df = country_df[country_df["selected_countries"].str.len() > 0].copy()

print(f"Số dòng sau lọc quốc gia: {len(filtered_country_df):,}")
print("\nPhân bố theo quốc gia chính:")
print(filtered_country_df["primary_selected_country"].value_counts())
display(filtered_country_df[["title", "production_countries", "selected_countries_text"]].head())

## 5. Lọc theo năm phát hành 2000–2025

Chuyển `release_date` sang datetime, tạo cột `release_year` và chỉ giữ phim phát hành trong giai đoạn 2000–2025.

> **Lưu ý:** Dữ liệu năm 2025 có thể chưa đầy đủ vì chưa phản ánh trọn năm, nên khi nhận xét xu hướng theo thời gian cần cẩn thận.

In [ ]:
# Chuyển ngày phát hành sang datetime và tạo năm phát hành.
filtered_country_df["release_date"] = pd.to_datetime(
    filtered_country_df["release_date"],
    errors="coerce",
)
filtered_country_df["release_year"] = filtered_country_df["release_date"].dt.year

invalid_release_date_count = filtered_country_df["release_date"].isna().sum()
print(f"Số dòng không parse được release_date: {invalid_release_date_count:,}")

# Lọc phim phát hành trong giai đoạn 2000–2025.
filtered_year_df = filtered_country_df[
    filtered_country_df["release_year"].between(2000, 2025, inclusive="both")
].copy()
filtered_year_df["release_year"] = filtered_year_df["release_year"].astype("Int64")

print(f"Số dòng sau lọc năm phát hành: {len(filtered_year_df):,}")

## 6. Lọc phim có dữ liệu chấm điểm và trạng thái phát hành

Chỉ giữ phim có `vote_average > 0` và `vote_count > 0`. Đồng thời kiểm tra trạng thái phát hành: ưu tiên chỉ giữ phim có trạng thái `Released`.

Nếu cột `status` không tồn tại trong dataset, notebook chỉ cảnh báo và bỏ qua bước lọc trạng thái, không gây lỗi.

In [ ]:
# Chuyển dữ liệu chấm điểm sang numeric để lọc chắc chắn.
filtered_year_df["vote_average"] = pd.to_numeric(filtered_year_df["vote_average"], errors="coerce")
filtered_year_df["vote_count"] = pd.to_numeric(filtered_year_df["vote_count"], errors="coerce")

# Chỉ giữ phim đã có điểm đánh giá và số lượt đánh giá lớn hơn 0.
filtered_score_df = filtered_year_df[
    (filtered_year_df["vote_average"].notna())
    & (filtered_year_df["vote_count"].notna())
    & (filtered_year_df["vote_average"] > 0)
    & (filtered_year_df["vote_count"] > 0)
].copy()

print(f"Số dòng sau lọc dữ liệu chấm điểm: {len(filtered_score_df):,}")

# Lọc trạng thái phát hành: chỉ giữ phim Released.
if "status" in filtered_score_df.columns:
    print("\nPhân bố trạng thái phim trước khi lọc:")
    print(filtered_score_df["status"].value_counts())

    before_status_filter = len(filtered_score_df)
    filtered_df = filtered_score_df[filtered_score_df["status"] == "Released"].copy()

    print(f"\nSố dòng trước lọc trạng thái: {before_status_filter:,}")
    print(f"Số dòng sau lọc trạng thái Released: {len(filtered_df):,}")
else:
    print("\n⚠️ Cảnh báo: Cột 'status' không tồn tại trong dataset. Bỏ qua bước lọc trạng thái.")
    filtered_df = filtered_score_df.copy()

print(f"\nSố dòng cuối cùng sau tất cả bộ lọc: {len(filtered_df):,}")

## 7. Đổi tên thuộc tính sang tiếng Việt không dấu

Sau khi lọc xong, các cột được đổi sang tên tiếng Việt không dấu để dễ sử dụng trong Power BI. Notebook chỉ đổi tên các cột đang tồn tại trong dataframe để tránh lỗi.

In [ ]:
# Mapping đổi tên cột sang tiếng Việt không dấu.
rename_mapping = {
    "id": "MaPhim",
    "title": "TenPhim",
    "vote_average": "DiemDanhGiaTB",
    "vote_count": "SoLuotDanhGia",
    "status": "TrangThai",
    "release_date": "NgayPhatHanh",
    "revenue": "DoanhThu",
    "runtime": "ThoiLuong",
    "adult": "PhimNguoiLon",
    "backdrop_path": "DuongDanAnhNen",
    "budget": "NganSach",
    "homepage": "TrangChu",
    "imdb_id": "MaIMDb",
    "original_language": "NgonNguGoc",
    "original_title": "TenGoc",
    "overview": "TomTatNoiDung",
    "popularity": "DoPhoBien",
    "poster_path": "DuongDanPoster",
    "tagline": "CauGioiThieu",
    "genres": "TheLoai",
    "production_companies": "CongTySanXuat",
    "production_countries": "QuocGiaSanXuat",
    "spoken_languages": "NgonNguDuocNoi",
    "keywords": "TuKhoa",
    "release_year": "NamPhatHanh",
    "selected_countries": "QuocGiaDuocChon",
    "selected_countries_text": "QuocGiaDuocChonText",
    "primary_selected_country": "QuocGiaChinh",
}

# Chỉ rename các cột tồn tại để tránh lỗi khi schema dữ liệu thay đổi.
existing_rename_mapping = {
    old_name: new_name
    for old_name, new_name in rename_mapping.items()
    if old_name in filtered_df.columns
}

filtered_df = filtered_df.rename(columns=existing_rename_mapping)

print("Danh sách cột sau khi đổi tên:")
print(filtered_df.columns.tolist())

## 8. Xử lý tên phim hiển thị cho dashboard

Tạo thêm cột `TenPhimHienThi` và cờ `CanBoSungTenTiengAnh` để phục vụ hiển thị trên dashboard.

**Logic:**
- Nếu `TenPhim` chứa ký tự Trung/Hàn/Nhật (CJK) → `CanBoSungTenTiengAnh = True`.
- Nếu `TenPhim` chỉ chứa ký tự Latin → `CanBoSungTenTiengAnh = False`.
- `TenPhimHienThi` tạm lấy giá trị từ `TenPhim` trong mọi trường hợp.

> **Lưu ý:** Không tự động dịch tên phim. Chỉ đánh dấu để review thủ công hoặc bổ sung bằng nguồn khác ở bước sau. Chỉ cần bổ sung tên tiếng Anh thủ công cho các phim xuất hiện trong top dashboard nếu cần.

In [ ]:
def contains_cjk(text):
    """Kiểm tra chuỗi có chứa ký tự Trung/Hàn/Nhật (CJK) hay không."""
    if not isinstance(text, str):
        return False
    # Phạm vi Unicode: CJK Unified Ideographs, CJK Extension A, Hangul, Hiragana, Katakana
    return bool(re.search(
        r'[\u4e00-\u9fff\u3400-\u4dbf\uac00-\ud7af\u3040-\u309f\u30a0-\u30ff]',
        text
    ))


# Tạo cột tên phim hiển thị (tạm giữ nguyên tên gốc).
filtered_df["TenPhimHienThi"] = filtered_df["TenPhim"]

# Đánh dấu phim có tên chứa ký tự CJK cần bổ sung tên tiếng Anh.
filtered_df["CanBoSungTenTiengAnh"] = filtered_df["TenPhim"].apply(contains_cjk)

# Thống kê phim cần bổ sung tên tiếng Anh.
need_english_count = filtered_df["CanBoSungTenTiengAnh"].sum()
print(f"Tổng số phim cần bổ sung tên tiếng Anh: {need_english_count:,}")

print("\nSố phim cần bổ sung tên tiếng Anh theo quốc gia:")
if need_english_count > 0:
    print(filtered_df[filtered_df["CanBoSungTenTiengAnh"]].groupby("QuocGiaChinh").size())
else:
    print("Không có phim nào cần bổ sung.")

print("\n20 dòng đầu:")
display_cols = ["MaPhim", "TenPhim", "TenGoc", "QuocGiaChinh", "NamPhatHanh", "CanBoSungTenTiengAnh"]
existing_display_cols = [c for c in display_cols if c in filtered_df.columns]
display(filtered_df[existing_display_cols].head(20))

## 9. Bổ sung cờ kiểm tra chất lượng dữ liệu

Tạo các cờ đánh dấu chất lượng dữ liệu mà **không xóa dòng nào**:

| Cờ | Ý nghĩa |
|---|---|
| `DuSoLuotDanhGia` | `True` nếu `SoLuotDanhGia >= 10` (ngưỡng tin cậy) |
| `CoDoanhThu` | `True` nếu `DoanhThu > 0` hợp lệ |
| `CoNganSach` | `True` nếu `NganSach > 0` hợp lệ |
| `ThoiLuongBatThuong` | `True` nếu `ThoiLuong > 300` phút |

> **Lưu ý:** Doanh thu và ngân sách thường thiếu nhiều trong TMDB, nên chỉ dùng cho phân tích phụ trên nhóm phim có dữ liệu hợp lệ. Không tự điền trung bình cho các giá trị thiếu.

In [ ]:
# --- Cờ đủ số lượt đánh giá (ngưỡng >= 10) ---
filtered_df["DuSoLuotDanhGia"] = filtered_df["SoLuotDanhGia"] >= 10

# --- Doanh thu: chuyển giá trị <= 0 thành NaN, tạo cờ ---
if "DoanhThu" in filtered_df.columns:
    filtered_df["DoanhThu"] = pd.to_numeric(filtered_df["DoanhThu"], errors="coerce")
    filtered_df.loc[filtered_df["DoanhThu"] <= 0, "DoanhThu"] = pd.NA
    filtered_df["CoDoanhThu"] = filtered_df["DoanhThu"].notna()
else:
    filtered_df["CoDoanhThu"] = False

# --- Ngân sách: chuyển giá trị <= 0 thành NaN, tạo cờ ---
if "NganSach" in filtered_df.columns:
    filtered_df["NganSach"] = pd.to_numeric(filtered_df["NganSach"], errors="coerce")
    filtered_df.loc[filtered_df["NganSach"] <= 0, "NganSach"] = pd.NA
    filtered_df["CoNganSach"] = filtered_df["NganSach"].notna()
else:
    filtered_df["CoNganSach"] = False

# --- Thời lượng: chuyển giá trị <= 0 thành NaN, đánh dấu outlier > 300 phút ---
if "ThoiLuong" in filtered_df.columns:
    filtered_df["ThoiLuong"] = pd.to_numeric(filtered_df["ThoiLuong"], errors="coerce")
    filtered_df.loc[filtered_df["ThoiLuong"] <= 0, "ThoiLuong"] = pd.NA
    filtered_df["ThoiLuongBatThuong"] = (
        filtered_df["ThoiLuong"].notna() & (filtered_df["ThoiLuong"] > 300)
    )
else:
    filtered_df["ThoiLuongBatThuong"] = False

# In thông tin tóm tắt các cờ.
print("Tóm tắt các cờ chất lượng dữ liệu:")
print(f"  Phim đủ lượt đánh giá (>= 10): {filtered_df['DuSoLuotDanhGia'].sum():,}")
print(f"  Phim có doanh thu hợp lệ:      {filtered_df['CoDoanhThu'].sum():,}")
print(f"  Phim có ngân sách hợp lệ:      {filtered_df['CoNganSach'].sum():,}")
print(f"  Phim có thời lượng hợp lệ:     {filtered_df['ThoiLuong'].notna().sum():,}")
print(f"  Phim có thời lượng bất thường:  {filtered_df['ThoiLuongBatThuong'].sum():,}")

# In danh sách phim có thời lượng bất thường để review.
outlier_runtime_df = filtered_df[filtered_df["ThoiLuongBatThuong"]]
if len(outlier_runtime_df) > 0:
    print("\nDanh sách phim có thời lượng bất thường (> 300 phút):")
    review_cols = ["MaPhim", "TenPhim", "ThoiLuong", "QuocGiaChinh", "NamPhatHanh"]
    existing_review_cols = [c for c in review_cols if c in filtered_df.columns]
    display(outlier_runtime_df[existing_review_cols].head(10))

## 10. Báo cáo kiểm tra chất lượng dữ liệu cuối cùng

Trước khi xuất file, in các thống kê tổng hợp để xác nhận dữ liệu đã lọc đáp ứng yêu cầu.

In [ ]:
initial_row_count = len(df)
country_filtered_row_count = len(filtered_country_df)
year_filtered_row_count = len(filtered_year_df)
scored_filtered_row_count = len(filtered_score_df)
final_row_count = len(filtered_df)
remaining_ratio = final_row_count / initial_row_count if initial_row_count else 0

print("=" * 60)
print("BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
print("=" * 60)

print(f"\n📊 Tổng quan lọc dữ liệu:")
print(f"  Tổng số dòng ban đầu:             {initial_row_count:,}")
print(f"  Sau lọc quốc gia:                  {country_filtered_row_count:,}")
print(f"  Sau lọc năm phát hành (2000–2025): {year_filtered_row_count:,}")
print(f"  Sau lọc dữ liệu chấm điểm:        {scored_filtered_row_count:,}")
print(f"  Sau lọc trạng thái Released:       {final_row_count:,}")
print(f"  Tỷ lệ dữ liệu còn lại:            {remaining_ratio:.2%}")

print(f"\n📊 Shape cuối cùng: {filtered_df.shape}")

# --- Số phim theo quốc gia ---
print("\n📊 Số phim theo quốc gia chính:")
country_counts = filtered_df["QuocGiaChinh"].value_counts()
print(country_counts)

# Ghi chú nếu lệch số lượng nhiều.
if "Vietnam" in country_counts.index:
    vn_count = country_counts.get("Vietnam", 0)
    max_count = country_counts.max()
    if vn_count < max_count * 0.3:
        print("\n⚠️ Số lượng phim Việt Nam ít hơn nhiều so với Hàn Quốc/Trung Quốc.")
        print("   → Khi so sánh nên dùng thêm tỷ lệ, trung bình, top phim thay vì chỉ số lượng tuyệt đối.")

# --- Số phim theo năm ---
print("\n📊 Số phim theo năm:")
year_counts = filtered_df["NamPhatHanh"].value_counts().sort_index()
print(year_counts.to_string())

# --- Số phim theo trạng thái ---
if "TrangThai" in filtered_df.columns:
    print("\n📊 Số phim theo trạng thái:")
    print(filtered_df["TrangThai"].value_counts())

# --- Thống kê các cờ chất lượng ---
print("\n📊 Thống kê cờ chất lượng dữ liệu:")
print(f"  Phim đủ lượt đánh giá (>= 10):    {filtered_df['DuSoLuotDanhGia'].sum():,} / {final_row_count:,}")
print(f"  Phim có doanh thu hợp lệ:          {filtered_df['CoDoanhThu'].sum():,} / {final_row_count:,}")
print(f"  Phim có ngân sách hợp lệ:          {filtered_df['CoNganSach'].sum():,} / {final_row_count:,}")
print(f"  Phim có thời lượng hợp lệ:         {filtered_df['ThoiLuong'].notna().sum():,} / {final_row_count:,}")
print(f"  Phim có thời lượng bất thường:      {filtered_df['ThoiLuongBatThuong'].sum():,}")
print(f"  Phim cần bổ sung tên tiếng Anh:    {filtered_df['CanBoSungTenTiengAnh'].sum():,}")

# --- Top 10 thể loại ---
if "TheLoai" in filtered_df.columns:
    print("\n📊 Top 10 thể loại nhiều phim nhất:")
    all_genres = (
        filtered_df["TheLoai"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
    )
    all_genres = all_genres[all_genres != ""]
    print(all_genres.value_counts().head(10))

# --- Thống kê nhanh điểm đánh giá, số lượt đánh giá, độ phổ biến ---
print("\n📊 Thống kê nhanh điểm đánh giá, số lượt đánh giá và độ phổ biến:")
summary_columns = ["DiemDanhGiaTB", "SoLuotDanhGia", "DoPhoBien"]
existing_summary_columns = [col for col in summary_columns if col in filtered_df.columns]
display(filtered_df[existing_summary_columns].describe())

# --- Một vài dòng dữ liệu mẫu ---
print("\n📊 Một vài dòng dữ liệu sau lọc:")
preview_columns = [
    "MaPhim",
    "TenPhim",
    "NgayPhatHanh",
    "NamPhatHanh",
    "QuocGiaDuocChonText",
    "DiemDanhGiaTB",
    "SoLuotDanhGia",
    "DoPhoBien",
]
existing_preview_columns = [col for col in preview_columns if col in filtered_df.columns]
display(filtered_df[existing_preview_columns].head())

## 11. Làm sạch ký tự xuống dòng đặc biệt

Một số trường văn bản trong TMDB (như `TomTatNoiDung`, `CauGioiThieu`) có thể chứa các ký tự xuống dòng đặc biệt:
- Line Separator (`\u2028`)
- Paragraph Separator (`\u2029`)
- Ký tự `\r`, `\n`, `\t` bên trong ô dữ liệu

Việc chuẩn hóa giúp file CSV ổn định hơn khi mở bằng VS Code và import vào Power BI, tránh cảnh báo "Detected unusual line terminators".

> **Lưu ý:** Chỉ xử lý các cột dạng text/object. Không ảnh hưởng đến cột số, ngày hay boolean. Không xóa dòng dữ liệu.

In [ ]:
def clean_text_value(value):
    """Thay thế các ký tự xuống dòng đặc biệt trong ô text và chuẩn hóa khoảng trắng."""
    if pd.isna(value):
        return value
    if not isinstance(value, str):
        return value

    # Thay các ký tự xuống dòng đặc biệt bằng khoảng trắng.
    value = value.replace("\u2028", " ")
    value = value.replace("\u2029", " ")
    value = value.replace("\r", " ")
    value = value.replace("\n", " ")
    value = value.replace("\t", " ")

    # Gom nhiều khoảng trắng liên tiếp thành 1 và strip đầu/cuối.
    value = re.sub(r"\s+", " ", value).strip()
    return value


# Chỉ làm sạch các cột dạng text/object.
text_cols = filtered_df.select_dtypes(include=["object"]).columns
print(f"Số cột text cần làm sạch: {len(text_cols)}")
print(f"Danh sách: {text_cols.tolist()}")

for col in text_cols:
    filtered_df[col] = filtered_df[col].apply(clean_text_value)

print("\n✅ Đã làm sạch ký tự xuống dòng đặc biệt trong các cột text.")

## 12. Xuất dữ liệu đã lọc

Dữ liệu sau lọc được lưu thành file CSV riêng trong `data/processed`. File gốc không bị ghi đè.

Các cột phục vụ tách bảng Dim/Fact/Bridge sau này được giữ lại: `QuocGiaDuocChon`, `QuocGiaDuocChonText`, `QuocGiaChinh`, `TheLoai`, `NgayPhatHanh`, `NamPhatHanh`.

In [ ]:
# Xuất dataframe cuối cùng ra file CSV phục vụ các bước tiếp theo.
filtered_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
)

print(f"✅ Đã lưu file: {OUTPUT_PATH}")
print(f"Shape cuối cùng: {filtered_df.shape}")
print(f"Số cột: {len(filtered_df.columns)}")
print(f"\nDanh sách cột trong file xuất:")
print(filtered_df.columns.tolist())

## 13. Các bước tiền xử lý tiếp theo

Các bước dưới đây chỉ là kế hoạch cho giai đoạn tiếp theo, chưa thực hiện trong notebook hiện tại. Bước tiếp theo sẽ là một notebook riêng để tách dữ liệu thành các bảng Dim/Fact/Bridge cho Power BI.

- Kiểm tra dữ liệu thiếu toàn diện.
- Chuẩn hóa kiểu dữ liệu.
- Xử lý giá trị không hợp lệ ở doanh thu, ngân sách, thời lượng.
- Tách bảng Phim (Dim_Phim).
- Tách bảng Hiệu suất phim (Fact_HieuSuat).
- Tạo bảng Thời gian (Dim_ThoiGian).
- Tạo bảng Quốc gia (Dim_QuocGia) và bảng liên kết Phim - Quốc gia (Bridge_Phim_QuocGia).
- Tạo bảng Thể loại (Dim_TheLoai) và bảng liên kết Phim - Thể loại (Bridge_Phim_TheLoai).
- Chuẩn bị dữ liệu cuối cùng để đưa vào Power BI.